# TPWRS Topic Clusters and Research-Agenda Affinity

For each topic cluster, this notebook shows the percentage of papers whose single highest adjudicated RA2025 score is Low, Moderate, or High.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.append(str(repo_root))

CLUSTER_PATH = repo_root / 'notebooks' / 'input' / 'IEEE_clustering' / 'TPWRS_2015_2025_clustering.csv'
RESULT_PATHS = sorted((repo_root / 'data' / 'results' / 'affinity_benchmark').glob('tpwrs_*/adjudicated_ra_affinities.csv'))
LEVEL_ORDER = ['Low', 'Moderate', 'High']
LEVEL_COLORS = {'Low': '#86b6ef', 'Moderate': '#2a78d6', 'High': '#104281'}

In [2]:
clusters = (
    pd.read_csv(CLUSTER_PATH, usecols=['doi', 'subject_2025'])
    .dropna(subset=['doi', 'subject_2025'])
    .drop_duplicates('doi')
    .rename(columns={'doi': 'Abstract_Index', 'subject_2025': 'Cluster'})
)

scores = pd.concat([pd.read_csv(path, usecols=['Abstract_Index', 'Final_Affinity']) for path in RESULT_PATHS])
paper_levels = (
    scores.groupby('Abstract_Index', as_index=False)['Final_Affinity'].max()
    .assign(Affinity_Level=lambda frame: pd.cut(
        frame['Final_Affinity'],
        bins=[float('-inf'), 40, 80, float('inf')],
        labels=LEVEL_ORDER,
    ))
)
cluster_levels = clusters.merge(paper_levels, on='Abstract_Index', how='inner')
print(f'{len(cluster_levels):,} clustered papers matched across {len(RESULT_PATHS)} adjudicated years.')

5,647 clustered papers matched across 12 adjudicated years.


In [3]:
cluster_mix = (
    cluster_levels.groupby(['Cluster', 'Affinity_Level'], observed=False).size()
    .rename('Papers').reset_index()
)
cluster_mix['Share'] = 100 * cluster_mix['Papers'] / cluster_mix.groupby('Cluster')['Papers'].transform('sum')
cluster_mix['Cluster'] = pd.Categorical(
    cluster_mix['Cluster'],
    categories=cluster_mix.groupby('Cluster')['Papers'].sum().sort_values().index,
    ordered=True,
)
cluster_mix.pivot(index='Cluster', columns='Affinity_Level', values='Share').fillna(0).round(1)

Affinity_Level,Low,Moderate,High
Cluster,,,
Resource Adequacy Planning,14.8,60.5,24.7
"Coupling, Hydrogen & Electrification",22.4,66.4,11.2
"Standards, Testing, Protection & Interconnection",12.5,70.0,17.5
"Resilience, Restoration & Extreme Events",0.6,63.2,36.3
GFM Inverters & IBR Control,5.6,42.9,51.5
"System Modelling, Digitalization & AI",20.2,69.0,10.8
Energy Storage,8.7,73.5,17.8
Renewable Forecasting & Digital Twins,4.2,69.3,26.5
"Transmission, HVDC & GETs",13.1,60.1,26.8


In [4]:
fig = px.bar(
    cluster_mix,
    x='Share',
    y='Cluster',
    color='Affinity_Level',
    orientation='h',
    barmode='stack',
    category_orders={'Affinity_Level': LEVEL_ORDER},
    color_discrete_map=LEVEL_COLORS,
    labels={'Share': 'Share of cluster papers (%)', 'Affinity_Level': 'Best RA affinity'},
    title='Best Research-Agenda affinity within each topic cluster',
)
fig.update_traces(texttemplate='%{x:.0f}%', textposition='inside')
fig.update_xaxes(range=[0, 100], ticksuffix='%')
fig.update_layout(height=max(450, 30 * cluster_mix['Cluster'].nunique()), legend_title_text='')
fig.show()

In [5]:
fig = px.bar(
    cluster_mix,
    x='Papers',
    y='Cluster',
    color='Affinity_Level',
    orientation='h',
    barmode='stack',
    category_orders={'Affinity_Level': LEVEL_ORDER},
    color_discrete_map=LEVEL_COLORS,
    labels={'Papers': 'Papers', 'Affinity_Level': 'Best RA affinity'},
    title='Research-Agenda affinity levels within each topic cluster',
)
fig.update_layout(height=max(450, 30 * cluster_mix['Cluster'].nunique()), legend_title_text='')
fig.show()

In [ ]:
from src.analysis import assign_prps

PROGRAMME_ORDER = ['IBR', 'Stability', 'DER', 'CROF', 'System Services', 'Planning', 'Other']
PROGRAMME_COLORS = {
    'IBR': '#2a78d6',
    'Stability': '#eb6834',
    'DER': '#1baf7a',
    'CROF': '#eda100',
    'System Services': '#e87ba4',
    'Planning': '#008300',
    'Other': '#898781',
}

programme_levels = assign_prps(pd.concat([pd.read_csv(path) for path in RESULT_PATHS], ignore_index=True))
paper_programmes = (
    programme_levels
    .sort_values(['Abstract_Index', 'Highest_Question_Affinity'], ascending=[True, False])
    .drop_duplicates('Abstract_Index')
    .assign(Programme=lambda frame: frame['PRP_Name'].where(frame['Affinity_Level'] != 'Low', 'Other'))
)
cluster_programme_counts = (
    clusters.merge(paper_programmes[['Abstract_Index', 'Programme']], on='Abstract_Index', how='inner')
    .groupby(['Cluster', 'Programme'], observed=True)
    .size().rename('Papers').reset_index()
)
cluster_programme_counts['Cluster'] = pd.Categorical(
    cluster_programme_counts['Cluster'],
    categories=cluster_mix['Cluster'].cat.categories,
    ordered=True,
)

fig = px.bar(
    cluster_programme_counts,
    x='Papers',
    y='Cluster',
    color='Programme',
    orientation='h',
    barmode='stack',
    category_orders={'Programme': PROGRAMME_ORDER},
    color_discrete_map=PROGRAMME_COLORS,
    labels={'Papers': 'Papers', 'Programme': 'Strongest Research Agenda programme'},
    title='Research-Agenda programme alignment within each topic cluster',
)
fig.update_layout(height=max(450, 30 * cluster_mix['Cluster'].nunique()), legend_title_text='')
fig.show()